# Мульти-продуктовый прогноз выручки — Prophet + GBM + автовыбор лучшей модели

**Источник данных один**: `data_multi_product_fact.xlsx`; train = все дни до 2026-04-01.

**Структура (три независимых блока):**
- **Часть 1 — Prophet**: Optuna по MAPE + guarded-бустер на остатках + якорь уровня с моментумом → `df_forecast_prophet.xlsx`
- **Часть 2 — GBM**: градиентный бустинг на лагах (log-цель, absolute-loss, праздничные признаки, рекурсивный прогноз) → `df_forecast_gbm.xlsx`
- **Часть 3 — сравнение и финальная сборка**: по каждому продукту автоматически выбирается лучшая модель **по бэктесту 90 дней** («в конце Q1 прогнозирую Q2»), победитель виден на графиках → `df_forecast_final.xlsx`, `comparison_final.xlsx`

Обе модели хорошо ловят тренд и сезонность, но разными механизмами: Prophet — явными компонентами (тренд + недельная/месячная/годовая сезонность + праздники), GBM — лагами 7–28 дней, которые автоматически адаптируются к недавним уровням, переломам и плато.

## Часть 1. Prophet + Optuna

In [ ]:
# =============================================================================
# Импорты
# =============================================================================
import contextlib
import logging
import os
import warnings

import holidays
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from optuna.samplers import TPESampler
from prophet import Prophet
from sklearn.metrics import (mean_absolute_error,
                             mean_absolute_percentage_error, r2_score)

# Бустер: LightGBM, при его отсутствии — тихий фолбэк на sklearn
try:
    from lightgbm import LGBMRegressor as BoosterModel
    BOOSTER_KW = dict(n_estimators=300, learning_rate=0.05,
                      num_leaves=31, verbose=-1)
except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor as BoosterModel
    BOOSTER_KW = dict(max_iter=300, learning_rate=0.05)

# =============================================================================
# Отключение логов — нужен только итоговый результат, не служебный вывод.
# cmdstanpy/Prophet вешают свой хендлер и ставят INFO ПРИ ИМПОРТЕ, а при каждом
# fit() могут повторить это. Фильтр на самом логгере отсекает записи ДО хендлеров,
# поэтому переживает и повторный setLevel(), и повторное добавление хендлера.
# =============================================================================
class _DropAll(logging.Filter):
    def filter(self, record):
        return False

def silence_loggers(names):
    for name in names:
        lg = logging.getLogger(name)
        lg.handlers.clear()
        lg.addFilter(_DropAll())
        lg.addHandler(logging.NullHandler())
        lg.setLevel(logging.CRITICAL)
        lg.propagate = False

silence_loggers(["prophet", "prophet.models", "cmdstanpy",
                 "optuna", "lightgbm"])
optuna.logging.set_verbosity(optuna.logging.ERROR)
warnings.filterwarnings("ignore")


In [ ]:
# ---- 1.0 данные для обучения ----
df_train = pd.read_excel("data_multi_product_fact.xlsx")
df_train = df_train[df_train["VALUE_DAY"] < "2026-04-01"]      # train до апреля 2026
df_train = df_train.rename(columns={"VALUE_DAY": "ds", "FEE_RUR_AMT_SUM": "y",
                                    "PRODUCT": "product"})
df_train["ds"] = pd.to_datetime(df_train["ds"])
df_train = (df_train.groupby(["product", "ds"], as_index=False)["y"].sum()
                    .sort_values(["product", "ds"]))

ds_min, ds_max = df_train["ds"].min().date(), df_train["ds"].max().date()
print(f"df_train: {len(df_train):,} строк, {ds_min} -> {ds_max}")


In [ ]:
# ---- 1.1 настройки ----
FUTURE_END    = "2026-12-31"    # горизонт прогноза
BT_DAYS       = 90              # бэктест-окно выбора модели: последние 90 дней train
                                # (обучаемся до него, прогнозируем его -> честный экзамен)
VAL_DAYS      = 92              # хвост для тюнинга Optuna ВНУТРИ бэктест-обучения
N_TRIALS      = 40              # попыток Optuna на продукт
MIN_HISTORY   = 180             # порог короткой истории (дней)
ANCHOR_DAYS   = 28              # окно якоря уровня
ANCHOR_CLIP   = (0.85, 1.15)    # пределы поправки уровня
MOMENTUM_CLIP = (0.99, 1.02)    # недельный темп продолжения текущего тренда
MOMENTUM_WEEKS = 8              # моментум действует первые N недель, дальше константа
TOTAL_CLIP    = (0.80, 1.40)    # предел суммарной поправки
SEED          = 42
FORECAST_XLSX = "df_forecast_prophet.xlsx"

HOLIDAYS_DF = pd.DataFrame(
    [{"ds": pd.to_datetime(d), "holiday": n, "lower_window": 0, "upper_window": 3}
     for d, n in holidays.Russia(
         years=range(df_train["ds"].dt.year.min(),
                     pd.Timestamp(FUTURE_END).year + 1)).items()])


In [ ]:
# ---- 1.2 календарные признаки ----
# сезонные регрессоры для Prophet: лето и зима
# (дневные/месячные паттерны уже покрыты сезонностями самой модели)
def build_regressors(df):
    df = df.copy()
    df["ds"] = pd.to_datetime(df["ds"])
    df["summer_season"] = df["ds"].dt.month.isin([6, 7, 8]).astype(int)
    df["winter_season"] = df["ds"].dt.month.isin([12, 1, 2]).astype(int)
    return df


# календарные признаки для бустера остатков (только циклические)
def booster_features(ds):
    ds = pd.to_datetime(ds)
    return pd.DataFrame({
        "dow":        ds.dt.dayofweek.values,
        "day":        ds.dt.day.values,
        "month":      ds.dt.month.values,
        "weekofyear": ds.dt.isocalendar().week.astype(int).values,
        "is_holiday": ds.isin(HOLIDAYS_DF["ds"]).astype(int).values,
    })


# IQR-клиппинг по дням недели. ТОЛЬКО для обучения, метрики — по сырым
def clip_outliers(df, iqr_mult=3.0):
    out = df.copy()
    dow = out["ds"].dt.dayofweek
    for d in range(7):
        vals = out.loc[dow == d, "y"]
        if len(vals) < 10:
            continue
        q1, q3 = vals.quantile([0.25, 0.75])
        out.loc[dow == d, "y"] = vals.clip(q1 - iqr_mult * (q3 - q1),
                                           q3 + iqr_mult * (q3 - q1))
    return out

# якорь уровня + моментум: поправка будущего по последним 28 дням обучения;
# используется одинаково в бэктесте и в финале, чтобы выбор отражал боевой конвейер
def apply_anchor(y_future, dates_future, df_hist, yhat_hist_tail):
    ratio  = (df_hist["y"].tail(ANCHOR_DAYS).values
              / np.maximum(np.asarray(yhat_hist_tail, dtype=float), 1e-9))
    anchor = float(np.clip(np.median(ratio), *ANCHOR_CLIP))
    mom    = float(np.clip((np.median(ratio[14:])
                            / max(np.median(ratio[:14]), 1e-9)) ** 0.5,
                           *MOMENTUM_CLIP))
    weeks  = ((pd.to_datetime(dates_future) - df_hist["ds"].max())
              .days / 7).values if not hasattr(dates_future, "dt") else \
             ((dates_future - df_hist["ds"].max()).dt.days / 7).values
    weeks  = np.clip(weeks, None, MOMENTUM_WEEKS)
    factor = np.clip(anchor * mom ** weeks, *TOTAL_CLIP)
    return np.clip(np.asarray(y_future, dtype=float) * factor, 0, None), anchor, mom


In [ ]:
# ---- 1.3 единственный переиспользуемый помощник: обучить Prophet и предсказать ----
import os
import contextlib

# глушим консольный вывод Stan (C-уровень) во время обучения Prophet
@contextlib.contextmanager
def _silence():
    with open(os.devnull, "w") as devnull:
        fd1, fd2 = os.dup(1), os.dup(2)
        try:
            os.dup2(devnull.fileno(), 1)
            os.dup2(devnull.fileno(), 2)
            yield
        finally:
            os.dup2(fd1, 1); os.dup2(fd2, 2)
            os.close(fd1);   os.close(fd2)

# Prophet на log1p(y): сезонности + праздники + регрессоры; прогноз в рублях
def fit_predict_prophet(train, predict_ds, params, yearly=True):
    tr = build_regressors(train)
    tr["y"] = np.log1p(tr["y"].clip(lower=0))
    model = Prophet(holidays=HOLIDAYS_DF, daily_seasonality=False,
                    weekly_seasonality=True, yearly_seasonality=yearly,
                    interval_width=0.95, **params)
    model.add_seasonality(name="monthly", period=30.5, fourier_order=5)
    if yearly:  # у короткой истории лето/зима — константы, регрессоры не нужны
        model.add_regressor("summer_season")
        model.add_regressor("winter_season")
    with _silence():
        model.fit(tr)
        pred = model.predict(build_regressors(predict_ds.copy()))
    pred["yhat"] = np.expm1(pred["yhat"])
    return model, pred

# якорь уровня + моментум: убирает смещение на границе train/прогноз и мягко
# продолжает текущий недельный темп последних 28 дней (все поправки с клипами)
def apply_anchor(forecast, train):
    tail     = train.tail(ANCHOR_DAYS).reset_index(drop=True)
    tail_fit = forecast.loc[forecast["ds"].isin(tail["ds"]), "yhat"].values
    ratio    = tail["y"].values / np.maximum(tail_fit, 1e-9)
    anchor   = float(np.clip(np.median(ratio), *ANCHOR_CLIP))
    mom      = float(np.clip((np.median(ratio[14:])
                              / max(np.median(ratio[:14]), 1e-9)) ** 0.5,
                             *MOMENTUM_CLIP))
    fut   = forecast["ds"] > train["ds"].max()
    weeks = ((forecast.loc[fut, "ds"] - train["ds"].max()).dt.days / 7
             ).clip(upper=MOMENTUM_WEEKS)
    factor = (anchor * mom ** weeks).clip(*TOTAL_CLIP)
    forecast.loc[fut, "yhat"] = forecast.loc[fut, "yhat"] * factor.values
    forecast["yhat"] = forecast["yhat"].clip(lower=0)
    return forecast

# ПОЛНЫЙ конвейер Prophet (guarded-бустер + якорь с моментумом).
# Используется дважды с одними параметрами: в бэктесте (train = данные до
# бэктест-окна) и в финале (train = весь train) — модель выбирается по тому же
# конвейеру, которым строится прогноз, без каких-либо утечек.
def prophet_pipeline(train, horizon, params):
    grid = pd.DataFrame({"ds": pd.date_range(train["ds"].min(),
                                             horizon["ds"].max(), freq="D")})
    # guarded-бустер: проверка на последних 60 днях train
    inner_cut = train["ds"].max() - pd.Timedelta(days=60)
    t0 = clip_outliers(train[train["ds"] <= inner_cut])
    v0 = train[train["ds"] > inner_cut]
    m0, p_v0 = fit_predict_prophet(t0, v0[["ds"]], dict(params))
    with _silence():
        p_t0 = m0.predict(build_regressors(t0[["ds"]].copy()))
    booster = BoosterModel(**BOOSTER_KW)
    booster.fit(booster_features(t0["ds"]),
                t0["y"].values - np.expm1(p_t0["yhat"].values))
    v0_boost = p_v0["yhat"].values + booster.predict(booster_features(v0["ds"]))
    use_booster = (mean_absolute_percentage_error(v0["y"], v0_boost.clip(0))
                   < mean_absolute_percentage_error(v0["y"], p_v0["yhat"]))

    # финальное обучение на всём train
    full = clip_outliers(train)
    model, forecast = fit_predict_prophet(full, grid, dict(params))
    if use_booster:
        with _silence():
            p_full = model.predict(build_regressors(full[["ds"]].copy()))
        booster = BoosterModel(**BOOSTER_KW)
        booster.fit(booster_features(full["ds"]),
                    full["y"].values - np.expm1(p_full["yhat"].values))
        forecast["yhat"] += booster.predict(booster_features(forecast["ds"]))

    forecast = apply_anchor(forecast, train)
    return forecast, use_booster


In [ ]:
# ---- 1.4 основной цикл: тюнинг -> честный бэктест 90 дней -> финальный прогноз ----
frames = []
for prod in sorted(df_train["product"].dropna().unique()):
    df = (df_train[df_train["product"] == prod]
          .sort_values("ds").reset_index(drop=True))
    short   = df["ds"].nunique() < MIN_HISTORY
    bt_days = 28 if short else BT_DAYS

    # бэктест-сплит: "в конце Q1 прогнозирую Q2" — модель выбирается по тому,
    # как она предсказала ПОСЛЕДНИЕ 90 дней, обучившись только на данных до них
    bt_cut = df["ds"].max() - pd.Timedelta(days=bt_days)
    sub = df[df["ds"] <= bt_cut].reset_index(drop=True)   # обучение бэктеста
    bt  = df[df["ds"] >  bt_cut].reset_index(drop=True)   # проверка бэктеста
    future = pd.DataFrame({"ds": pd.date_range(df["ds"].min(), FUTURE_END,
                                               freq="D")})
    print(f"▶ {prod}: tuning + backtest + forecast ...", end=" ", flush=True)

    if short:
        # упрощённая ветка: недельная + месячная сезонность, без Optuna/бустера
        _, p_bt = fit_predict_prophet(sub, bt[["ds"]], {}, yearly=False)
        bt_mape = mean_absolute_percentage_error(
            bt["y"], p_bt["yhat"].clip(lower=0)) * 100
        model, forecast = fit_predict_prophet(df, future, {}, yearly=False)
        forecast = apply_anchor(forecast, df)
        best, use_booster = {}, False

    else:
        # --- Optuna: тюнинг ТОЛЬКО на данных до бэктеста (никакой утечки
        # в окно, по которому потом выбирается модель) ---
        val_cutoff = sub["ds"].max() - pd.Timedelta(days=VAL_DAYS)
        tr  = clip_outliers(sub[sub["ds"] <= val_cutoff])
        val = sub[sub["ds"] > val_cutoff]

        def objective(trial):
            params = {
                "changepoint_prior_scale": trial.suggest_float(
                    "changepoint_prior_scale", 1e-3, 0.5, log=True),
                "seasonality_prior_scale": trial.suggest_float(
                    "seasonality_prior_scale", 0.01, 10.0, log=True),
                "holidays_prior_scale": trial.suggest_float(
                    "holidays_prior_scale", 0.01, 10.0, log=True),
                "seasonality_mode": trial.suggest_categorical(
                    "seasonality_mode", ["additive", "multiplicative"]),
                "n_changepoints": trial.suggest_int("n_changepoints", 15, 40),
                "changepoint_range": trial.suggest_float(
                    "changepoint_range", 0.80, 0.92),
            }
            _, pred = fit_predict_prophet(tr, val[["ds"]], params)
            return mean_absolute_percentage_error(val["y"], pred["yhat"])

        study = optuna.create_study(direction="minimize",
                                    sampler=TPESampler(seed=SEED))
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
        best = study.best_params

        # --- честный бэктест: ПОЛНЫЙ конвейер на sub -> прогноз 90 дней ---
        fc_bt, _ = prophet_pipeline(sub, bt[["ds"]], best)
        p_bt = fc_bt.loc[fc_bt["ds"].isin(bt["ds"]), "yhat"].values
        bt_mape = mean_absolute_percentage_error(bt["y"], p_bt) * 100

        # --- финальный прогноз: тот же конвейер на всём train ---
        forecast, use_booster = prophet_pipeline(df, future, best)

    print(f"готово (бэктест MAPE {bt_mape:.1f}%, "
          f"бустер {'ON' if use_booster else 'off'}"
          + (", короткая история" if short else "") + ")")

    out = forecast[["ds", "yhat"]].copy()
    out.insert(0, "product", prod)
    out["train_end"] = df["ds"].max()
    out["booster"]   = "yes" if use_booster else "no"
    out["bt_mape"]   = round(bt_mape, 2)     # оценка для автовыбора в Части 3
    frames.append(out)


In [ ]:
# ---- 1.5 сборка и сохранение ----
df_forecast_prophet = pd.concat(frames, ignore_index=True)
df_forecast_prophet.to_excel(FORECAST_XLSX, index=False)
n_products = df_forecast_prophet["product"].nunique()
print(f"\ndf_forecast_prophet: {len(df_forecast_prophet):,} строк, "
      f"{n_products} продуктов -> {FORECAST_XLSX}")


## Часть 2. GBM — градиентный бустинг с лагами

Проверенная на факте конфигурация: **log1p-цель** (относительные ошибки → MAPE),
**absolute-loss** (устойчивость к выбросам), **праздничные признаки** (день праздника
и предпраздничный день), лаги 7/14/21/28 и скользящие средние 7/28. Годовой лаг 364
проверялся и отклонён: на валидации он выглядит хорошо, а на реальном OOS ломает
продукты с переломами тренда (Выплаты, ИЭ AFT) — годовую сезонность закрывает Prophet.
Прогноз рекурсивный: свой прогноз подставляется как лаг для следующих дней.

In [ ]:
# ---- 2.0 данные и настройки (Часть 2 автономна) ----
import numpy as np
import pandas as pd
import holidays as holidays_lib
from sklearn.metrics import mean_absolute_percentage_error
import warnings
warnings.filterwarnings("ignore")

# --- отключение логов (Часть 2 автономна) ---
import logging
class _DropAllGBM(logging.Filter):
    def filter(self, record):
        return False
for _name in ("lightgbm", "cmdstanpy", "prophet"):
    _lg = logging.getLogger(_name)
    _lg.handlers.clear()
    _lg.addFilter(_DropAllGBM())
    _lg.addHandler(logging.NullHandler())
    _lg.setLevel(logging.CRITICAL)
    _lg.propagate = False
warnings.filterwarnings("ignore")

try:
    from lightgbm import LGBMRegressor as GBMModel
    GBM_KW = dict(n_estimators=400, learning_rate=0.05, num_leaves=31,
                  objective="mae", verbose=-1)
except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor as GBMModel
    GBM_KW = dict(max_iter=400, learning_rate=0.05, loss="absolute_error")

GBM_SOURCE  = "data_multi_product_fact.xlsx"
GBM_CUTOFF  = "2026-04-01"        # train до апреля 2026 (как у Prophet)
GBM_END     = "2026-12-31"        # горизонт прогноза
GBM_XLSX    = "df_forecast_gbm.xlsx"
GBM_LAGS    = [7, 14, 21, 28]
GBM_ROLLS   = [7, 28]

df_gbm = pd.read_excel(GBM_SOURCE)
df_gbm = df_gbm[df_gbm["VALUE_DAY"] < GBM_CUTOFF]
df_gbm = df_gbm.rename(columns={"VALUE_DAY": "ds", "FEE_RUR_AMT_SUM": "y",
                                "PRODUCT": "product"})
df_gbm["ds"] = pd.to_datetime(df_gbm["ds"])
df_gbm = (df_gbm.groupby(["product", "ds"], as_index=False)["y"].sum()
                .sort_values(["product", "ds"]))

RU_HOLIDAYS = set(pd.to_datetime(list(holidays_lib.Russia(
    years=range(df_gbm["ds"].dt.year.min(),
                pd.Timestamp(GBM_END).year + 1)).keys())))
print(f"train: {len(df_gbm):,} строк, до {df_gbm['ds'].max().date()}; "
      f"праздников в календаре: {len(RU_HOLIDAYS)}")

In [ ]:
# ---- 2.1 модель GBM ----
FEAT_ORDER = (["dow", "day", "month", "is_hol", "pre_hol"]
              + [f"lag{L}" for L in GBM_LAGS]
              + [f"roll{R}" for R in GBM_ROLLS])

# вектор признаков одной даты (s — серия log1p-значений с DatetimeIndex)
def gbm_feats(d, s):
    row = [d.dayofweek, d.day, d.month,
           int(d in RU_HOLIDAYS), int((d + pd.Timedelta(days=1)) in RU_HOLIDAYS)]
    for L in GBM_LAGS:
        row.append(s.get(d - pd.Timedelta(days=L), np.nan))
    tail = s.loc[:d - pd.Timedelta(days=1)]
    for R in GBM_ROLLS:
        row.append(tail.tail(R).mean())
    return row

# обучение на истории + рекурсивный прогноз; возвращает (in-sample, будущее)
def gbm_predict(tr, dates):
    tgt = tr.set_index("ds")["y"].astype(float)
    tgt = pd.Series(np.log1p(tgt.values), index=tgt.index)      # log-цель
    warm = max(GBM_LAGS)
    X = np.array([gbm_feats(d, tgt) for d in tgt.index[warm:]])
    model = GBMModel(random_state=42, **GBM_KW)
    model.fit(X, tgt.values[warm:])

    hist_pred = np.concatenate([tr["y"].values[:warm],            # прогрев — фактом
                                np.expm1(model.predict(X)).clip(0)])
    s, out = tgt.copy(), []
    for d in dates:                                               # рекурсия
        p = float(model.predict(np.array([gbm_feats(d, s)]))[0])
        s.loc[d] = p
        out.append(max(np.expm1(p), 0.0))
    return hist_pred, np.array(out)

In [ ]:
# ---- 2.2 цикл: честный бэктест 90 дней -> финальный прогноз ----
gbm_frames = []
for prod in sorted(df_gbm["product"].dropna().unique()):
    d = df_gbm[df_gbm["product"] == prod].sort_values("ds").reset_index(drop=True)
    print(f"▶ {prod}: gbm ...", end=" ", flush=True)

    # бэктест "в конце Q1 прогнозирую Q2": обучение до последних 90 дней train,
    # прогноз этих 90 дней — то же окно и тот же принцип, что у Prophet
    bt_days = 90 if d["ds"].nunique() >= 180 else 28
    bt_cut  = d["ds"].max() - pd.Timedelta(days=bt_days)
    sub, bt = d[d["ds"] <= bt_cut], d[d["ds"] > bt_cut].reset_index(drop=True)
    _, p_bt = gbm_predict(sub, bt["ds"])
    bt_mape = mean_absolute_percentage_error(bt["y"], p_bt) * 100

    # финальное обучение на всём train, прогноз до GBM_END
    grid = pd.date_range(d["ds"].min(), GBM_END, freq="D")
    fut  = pd.DatetimeIndex([x for x in grid if x > d["ds"].max()])
    hist_pred, fut_pred = gbm_predict(d, fut)

    gbm_frames.append(pd.DataFrame({
        "product": prod, "ds": grid,
        "yhat": np.concatenate([hist_pred, fut_pred]),
        "train_end": d["ds"].max(),
        "bt_mape": round(bt_mape, 2)}))
    print(f"готово (бэктест MAPE {bt_mape:.1f}%)")

df_forecast_gbm = pd.concat(gbm_frames, ignore_index=True)
df_forecast_gbm.to_excel(GBM_XLSX, index=False)
print(f"\ndf_forecast_gbm: {len(df_forecast_gbm):,} строк -> {GBM_XLSX}")


## Часть 3. Сравнение и финальная сборка

Читает оба прогноза и факт. По каждому продукту **автоматически выбирается модель
с меньшим MAPE на бэктесте 90 дней**: обе модели полным конвейером обучаются на данных до последних 90 дней train и прогнозируют их — ровно как в бою («в конце Q1 прогнозирую Q2»).
Финальный прогноз (`df_forecast_final.xlsx`) собирается из победителей; на графиках
победитель выделен и указан в заголовке. OOS-метрики считаются для обеих моделей
и для финальной сборки.

In [ ]:
# ---- 3.0 загрузка и объединение ----
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import (mean_absolute_error,
                             mean_absolute_percentage_error, r2_score)

fc_p = pd.read_excel("df_forecast_prophet.xlsx")
fc_g = pd.read_excel("df_forecast_gbm.xlsx")
for f in (fc_p, fc_g):
    f["ds"] = pd.to_datetime(f["ds"])
    f["train_end"] = pd.to_datetime(f["train_end"])

fact = pd.read_excel("data_multi_product_fact.xlsx").rename(
    columns={"VALUE_DAY": "ds", "FEE_RUR_AMT_SUM": "y", "PRODUCT": "product"})
fact["ds"] = pd.to_datetime(fact["ds"])
fact = fact.groupby(["product", "ds"], as_index=False)["y"].sum()

df_cmp = (fc_p[["product", "ds", "yhat", "train_end", "bt_mape"]]
          .rename(columns={"yhat": "yhat_prophet", "bt_mape": "bt_prophet"})
          .merge(fc_g[["product", "ds", "yhat", "bt_mape"]]
                 .rename(columns={"yhat": "yhat_gbm", "bt_mape": "bt_gbm"}),
                 on=["product", "ds"], how="left")
          .merge(fact, on=["product", "ds"], how="left"))
df_cmp["is_oos"] = (df_cmp["ds"] > df_cmp["train_end"]) & df_cmp["y"].notna()
print(f"строк: {len(df_cmp):,}, OOS: {df_cmp['is_oos'].sum()}")

In [ ]:
# ---- 3.1 автовыбор победителя (по бэктесту 90 дней) и финальная сборка ----
bt_scores = (df_cmp.groupby("product")[["bt_prophet", "bt_gbm"]].first())
bt_scores["winner"] = np.where(bt_scores["bt_gbm"] < bt_scores["bt_prophet"],
                               "gbm", "prophet")
winners = bt_scores["winner"].to_dict()

df_cmp["winner"] = df_cmp["product"].map(winners)
df_cmp["yhat_final"] = np.where(df_cmp["winner"] == "gbm",
                                df_cmp["yhat_gbm"], df_cmp["yhat_prophet"])

df_forecast_final = df_cmp[["product", "ds", "yhat_final", "winner", "train_end"]]
df_forecast_final = df_forecast_final.rename(columns={"yhat_final": "yhat",
                                                      "winner": "model"})
df_forecast_final.to_excel("df_forecast_final.xlsx", index=False)

# OOS-метрики: prophet / gbm / финальная сборка
rows = []
for prod, g in df_cmp[df_cmp["is_oos"]].groupby("product"):
    row = {"product": prod, "oos_days": len(g), "winner_bt": winners[prod],
           "bt_prophet_%": bt_scores.loc[prod, "bt_prophet"],
           "bt_gbm_%": bt_scores.loc[prod, "bt_gbm"]}
    for m in ("prophet", "gbm", "final"):
        yh = g[f"yhat_{m}"]
        row[f"MAPE_{m}_%"] = mean_absolute_percentage_error(g["y"], yh) * 100
        row[f"diff_{m}_%"] = (yh.sum() / g["y"].sum() - 1) * 100
    rows.append(row)
final_metrics = pd.DataFrame(rows).sort_values("MAPE_final_%").reset_index(drop=True)

with pd.ExcelWriter("comparison_final.xlsx") as xl:
    df_cmp.to_excel(xl, sheet_name="daily", index=False)
    final_metrics.to_excel(xl, sheet_name="metrics_oos", index=False)
print("Сохранено -> df_forecast_final.xlsx, comparison_final.xlsx")
show_cols = ["product", "winner_bt", "bt_prophet_%", "bt_gbm_%",
             "MAPE_prophet_%", "MAPE_gbm_%", "MAPE_final_%"]
display(final_metrics[show_cols].round(1))


In [ ]:
# ---- 3.2 графики: факт vs обе модели, победитель выделен ----
CLR = {"fact": "#21364B", "prophet": "#C8102E", "gbm": "#E67E22"}
last_m = fact["ds"].max().to_period("M")          # неполный месяц — вне графика

for prod, g in df_cmp.groupby("product"):
    train_end, winner = g["train_end"].iloc[0], winners[prod]
    met = final_metrics[final_metrics["product"] == prod]

    m = g.copy()
    m["ym"] = m["ds"].dt.to_period("M")
    mm = m.groupby("ym").agg(prophet=("yhat_prophet", "sum"),
                             gbm=("yhat_gbm", "sum"),
                             fact=("y", lambda s: s.sum(min_count=1))).reset_index()
    mm.loc[mm["ym"] == last_m, "fact"] = np.nan
    mm["ym"] = mm["ym"].dt.to_timestamp()

    fig, ax = plt.subplots(figsize=(13, 5))
    title = f"{prod}   |   лучшая модель: {winner.upper()} (по бэктесту 90 дн.)"
    if len(met):
        title += f"   |   MAPE OOS финала: {met['MAPE_final_%'].iloc[0]:.1f}%"
    ax.set_title(title, fontsize=13, fontweight="bold", loc="left")

    loser = "gbm" if winner == "prophet" else "prophet"
    ax.plot(mm["ym"], mm[loser] / 1e9, color=CLR[loser], lw=1.2, ls="--",
            alpha=0.5, marker="^", ms=3, label=loser, zorder=2)
    ax.plot(mm["ym"], mm[winner] / 1e9, color=CLR[winner], lw=2.4, ls="--",
            marker="s", ms=4, label=f"{winner} (winner)", zorder=3)
    ax.plot(mm["ym"], mm["fact"] / 1e9, color=CLR["fact"], lw=2.4, marker="o",
            ms=4, label="факт", zorder=4)

    ax.axvline(train_end, color="gray", ls=":", lw=1.5)
    ax.text(train_end, 0.98, " конец train", transform=ax.get_xaxis_transform(),
            fontsize=9, color="gray", va="top")
    ax.set_ylabel("млрд ₽")
    ax.legend(loc="upper left", frameon=True)
    ax.grid(alpha=0.3)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
# ---- 3.3 итог по портфелю ----
print("Финальная сборка (сортировка по MAPE):\n")
for _, r in final_metrics.iterrows():
    flag = "OK " if r["MAPE_final_%"] < 15 else ("~  " if r["MAPE_final_%"] < 30 else "!! ")
    print(f"  {flag}{r['product']:<26} {r['winner_bt']:<8} "
          f"MAPE={r['MAPE_final_%']:5.1f}%  перекос {r['diff_final_%']:+.1f}%")

oos = df_cmp[df_cmp["is_oos"]]
tf = oos["y"].sum()
print(f"\nПерекос суммы портфеля OOS: prophet {(oos['yhat_prophet'].sum()/tf-1)*100:+.1f}% | "
      f"gbm {(oos['yhat_gbm'].sum()/tf-1)*100:+.1f}% | "
      f"ФИНАЛ {(oos['yhat_final'].sum()/tf-1)*100:+.1f}%")
n_g = sum(1 for w in winners.values() if w == "gbm")
print(f"Победители: GBM {n_g} продуктов, Prophet {len(winners)-n_g}")